# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahmoud-Beram/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type:** Scoring / Regression (not classification).

My lane (Refresh / Content Opportunity Scoring) needs a continuous ranked score, not a
yes/no label — editors have limited time and need to know *which* pages to review first,
not just *whether* a page is "bad." I predict a continuous target (actual CTR, or the gap
between expected and actual CTR) and rank all pages by it. This is closer to a ranking/
scoring problem than classification, because the deliverable is an ordered queue, not a
category.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `actual_ctr = total_clicks / total_imp`, aggregated per (client, content) for
a given month.

**Where the label comes from:** an observed outcome, not a defined rule — it's computed
directly from real GSC clicks and impressions, so it's not a proxy in the sense of being
hand-picked. The derived decision metric, `lost_clicks = (predicted_ctr - actual_ctr) *
total_imp`, ranks pages by how many clicks the model expects them to be missing given
their position, content age, and content properties.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** R² for raw predictive fit, but the metric that actually matters for the
lane is Precision@K on an independently-defined "truly underperforming" label (bottom
25% CTR within its position bucket — defined separately from the ranking score itself,
to avoid a circular evaluation).

**What "good" means:** measured honestly (after removing a temporal leak found in
validation), the model reaches R²≈0.12 on a random split and R²≈0.08 on a time-based
split for clients with existing history — modest but real, and clearly better than the
position-only baseline (R²≈0.06). It fails for entirely new clients (R²<0, grouped
split) — a scoped limitation, not a claim of universal accuracy.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb, pandas as pd
import warnings
warnings.filterwarnings('ignore')

hf_token = userdata.get('HF_TOKEN').strip()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

print("Loading feature dataset...")

query = f"""
    WITH perf AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS total_imp, SUM(gsc_clicks) AS total_clicks,
               AVG(gsc_avg_position) AS avg_pos
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
        HAVING total_imp >= 500
    )
    SELECT
        p.*,
        c.word_count, c.search_volume, c.competition, c.cpc, c.backlinks,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
    FROM perf p
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
        ON p.content_hash_id = c.content_hash_id
    WHERE c.is_published = TRUE AND c.is_deleted = FALSE
"""

df = con.execute(query).df()
df['actual_ctr'] = df['total_clicks'] / df['total_imp']

print(f"Dataset ready: {len(df)} articles.")
print(list(df.columns))

Loading feature dataset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset ready: 61911 articles.
['client_hash_id', 'content_hash_id', 'total_imp', 'total_clicks', 'avg_pos', 'word_count', 'search_volume', 'competition', 'cpc', 'backlinks', 'content_age_days', 'actual_ctr']


In [3]:
# Unit of analysis: one row = one (client, content_id) pair's aggregated performance
# for a given month.
print(df[['client_hash_id', 'content_hash_id', 'avg_pos', 'total_imp',
           'word_count', 'content_age_days', 'actual_ctr']].head())
print(f"\nRows: {len(df)} | Unique clients: {df['client_hash_id'].nunique()} | "
      f"Unique content items: {df['content_hash_id'].nunique()}")

            client_hash_id           content_hash_id   avg_pos  total_imp  \
0  client_73cda7b4e4f265ea  content_7a105f548d9c6916  7.209549     6523.0   
1  client_73cda7b4e4f265ea  content_36c36abc7650d7af  6.724039     5630.0   
2  client_73cda7b4e4f265ea  content_a7da352b73b02668  7.244844     4944.0   
3  client_73cda7b4e4f265ea  content_aafb2ab7e5fc80d0  5.258331     7709.0   
4  client_73cda7b4e4f265ea  content_20403327d8d9374c  8.834415     3561.0   

   word_count  content_age_days  actual_ctr  
0        2123               396    0.001073  
1        2546               396    0.001066  
2        2330               396    0.002629  
3        2556               396    0.002594  
4        3010               396    0.002808  

Rows: 61911 | Unique clients: 36 | Unique content items: 61911


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "impressions > 1000 AND position ≤ 10 AND CTR < 1%" — our Week-4
baseline) only flags pages that violate a single hard threshold. It found ~28,000 of
~62,000 eligible pages, but treats them all equally, with no sense of *how much* opportunity
each represents. Position, content age, word count, and SEO metadata interact — a mildly
stale page in a high-impression slot may matter more than a very old page with low
impressions — and a single if-statement can't rank that trade-off. Our own Precision@K
test (Week 5/6) confirmed this: the learned model outperformed the Week-4 rule across
K=10–50, precisely because it weighs multiple continuous signals instead of a hard cutoff.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.